In [1]:
import pandas as pd
from opensearchpy import OpenSearch
from opensearchpy import helpers
from sentence_transformers import SentenceTransformer

In [2]:
host = 'some.host.com'
port = 3005
auth = ('joedoe', 'qwerty') 

client = OpenSearch(
    hosts = [{'host': host, 'port': port}],
    http_auth = auth,
    use_ssl = True,
    verify_certs = False,
    ssl_assert_hostname = False,
    ssl_show_warn = False,
)

# I am using the same model as deployed in opensearch
model_name = "msmarco-distilbert-base-tas-b"
model = SentenceTransformer(model_name)

EMBEDDING_DIM = model.encode(["Sample sentence"])[0].shape[0]
print(EMBEDDING_DIM)


768


In [3]:
file_path = './archive/tmdb_5000_movies.csv'

# Specifying the columns we want to include in the DataFrame
columns = ['id', 'original_title', 'overview']

# Load the CSV file into a DataFrame with selected columns
df_movies = pd.read_csv(file_path, usecols=columns)

# Remove rows with NaN values
df_movies = df_movies.dropna()

# Display the DataFrame
print(df_movies.head(5))

       id                            original_title  \
0   19995                                    Avatar   
1     285  Pirates of the Caribbean: At World's End   
2  206647                                   Spectre   
3   49026                     The Dark Knight Rises   
4   49529                               John Carter   

                                            overview  
0  In the 22nd century, a paraplegic Marine is di...  
1  Captain Barbossa, long believed to be dead, ha...  
2  A cryptic message from Bond’s past sends him o...  
3  Following the death of District Attorney Harve...  
4  John Carter is a war-weary, former military ca...  


In [4]:
# now create vector indices from scratch
index_name = "movies"

try:
    response = client.indices.delete(index=index_name) 
    print(response)
except:
    pass

index_body = {
  "settings": {
    "index.knn": True
  },
  "mappings": {
    "properties": {
      "id": {
        "type": "text"
      },
      "embedding": {
        "type": "knn_vector",
        "dimension": EMBEDDING_DIM,
        "method": {
          "engine": "lucene",
          "space_type": "l2",
          "name": "hnsw",
          "parameters": {}
        }
      },
      "original_title": {
        "type": "text"
      },
      "overview": {
        "type": "text"
      }
    }
  }
}

response = client.indices.create(index=index_name, body=index_body)  
print(response)


{'acknowledged': True}
{'acknowledged': True, 'shards_acknowledged': True, 'index': 'movies'}


In [5]:
# add documents to the vector index

df_movies = df_movies.head(10)

# Get embeddings using the model
embeddings = model.encode(df_movies["overview"])

df = pd.DataFrame({
    "id": df_movies["id"].tolist(),
    "original_title": df_movies["original_title"].tolist(),
    "overview": df_movies["overview"].tolist(),
    "embedding": embeddings.tolist(),  # Convert embedding to list for DataFrame,
})

docs = df.to_dict(orient="records")

#helpers.bulk(client, docs, index=index_name, raise_on_error=True, refresh=True)

for index, row in df_movies.iterrows():
    print(f"Id: {row['id']}, Title: {row['original_title']}, Overview: {row['overview']}")
    original_title = row['original_title']
    overview = row['overview']
    id = row['id']

    # Sentence transformer model takes list of documents as input and returns list of embeddings.
    embedding = list(model.encode([overview])[0])
    # We are inserting a data point with 3 attribute, "id", "text" and "embedding" as knn_vector type.
    my_doc = {"id": id, "original_title": original_title, "overview": overview, "embedding": embedding}
    res = client.index(
        index=index_name,
        body=my_doc,
        id = str(index),
        refresh = True
    )
    print(res)


Id: 19995, Title: Avatar, Overview: In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.
{'_index': 'movies', '_id': '0', '_version': 1, 'result': 'created', 'forced_refresh': True, '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 0, '_primary_term': 1}
Id: 285, Title: Pirates of the Caribbean: At World's End, Overview: Captain Barbossa, long believed to be dead, has come back to life and is headed to the edge of the Earth with Will Turner and Elizabeth Swann. But nothing is quite as it seems.
{'_index': 'movies', '_id': '1', '_version': 1, 'result': 'created', 'forced_refresh': True, '_shards': {'total': 2, 'successful': 1, 'failed': 0}, '_seq_no': 1, '_primary_term': 1}
Id: 206647, Title: Spectre, Overview: A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles political forces to keep the secr

In [7]:
# Ask some queries using knn search
user_query = "A spy goes on a mission"

""" Embedding the query by using the same model """
query_embedding = list(model.encode((user_query)))

print(query_embedding[:20])

query_body = {
  "_source": {
    "excludes": [
      "embedding"
    ]
  },
  "query": {
    "knn": {
      "embedding": {
        "vector":query_embedding,
        "k": 5
      }
    }
  }
}

index_name = 'movies'

results = client.search(
    body=query_body,
    index=index_name
)

print(results)

[-0.13026899, 0.038367104, -0.29232928, 0.25385237, 0.3061329, -0.19019394, 0.5934607, -0.22599499, -0.0952346, -0.13156211, -0.4465393, -0.0139138885, -0.07234843, -0.09539731, 0.36534357, -0.18958792, 0.522665, -0.5005499, 0.29866576, -0.30213106]
{'took': 19, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 5, 'relation': 'eq'}, 'max_score': 0.017188461, 'hits': [{'_index': 'movies', '_id': '2', '_score': 0.017188461, '_source': {'overview': 'A cryptic message from Bond’s past sends him on a trail to uncover a sinister organization. While M battles political forces to keep the secret service alive, Bond peels back the layers of deceit to reveal the terrible truth behind SPECTRE.', 'original_title': 'Spectre', 'id': 206647}}, {'_index': 'movies', '_id': '0', '_score': 0.013856831, '_source': {'overview': 'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn 